# 🌿 Plant Disease Detector v3: better on real-world photos

**Problem with v2:** 91.6% on PlantVillage lab photos, but only 2/12 exact on real garden photos. Every PlantVillage
photo is one leaf on a plain grey background, so the model learned that background.

**What v3 changes**
1. **Real photos in training:** the PlantDoc dataset (about 2,600 real-world photos from the web, CC BY 4.0) is
   mixed into training and repeated 6× so it carries real weight.
2. **Background swaps:** half of the PlantVillage leaves are cut out (using PlantVillage's segmented versions) and
   pasted onto a random real background taken from PlantDoc photos.
3. **Stronger augmentation:** rotation, zoom, shift, brightness, contrast and blur.
4. **Higher resolution:** 224 px instead of 160 px, and longer fine-tuning.

**Fair, honest testing**
- (a) the **same 190 lab photos** as v2 (same seed);
- (b) **PlantDoc's own test split** (never used for training);
- (c) the **same 12 Wikimedia Commons photos** used to test v2.

Training photos that are near-duplicates of any test photo are **removed** first. v2 is downloaded from the
website and scored on exactly the same tests.

**Before you start:** Runtime → Change runtime type → **T4 GPU**; Colab secret `KAGGLE_API_TOKEN` (🔑, Notebook access on).

**Disconnect-proof:** the notebook saves a checkpoint to Google Drive (`MyDrive/plant-disease-v3/`) after every epoch.
If Colab disconnects, just reconnect and **Run all** again: the data is rebuilt the same way (fixed seeds), and training
**continues from the last saved epoch** instead of starting over. The final model and test report are also saved to Drive.

## 1. Google Drive (checkpoints) and Kaggle setup

In [ ]:
!pip -q install -U "kaggle>=1.8"
import os, shutil, json, pathlib
from google.colab import drive, files, userdata

drive.mount('/content/drive')
CKPT = pathlib.Path('/content/drive/MyDrive/plant-disease-v3')
CKPT.mkdir(parents=True, exist_ok=True)
STATE_FILE = CKPT / 'state.json'
state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {'stage': 1, 'epoch': 0, 'best': -1.0, 'wait': 0, 'history': {}, 'stage1_epochs': 0}
def save_state():
    STATE_FILE.write_text(json.dumps(state, indent=1))
print('Checkpoints folder:', CKPT)
print('Starting fresh' if state['stage'] == 1 and state['epoch'] == 0 else f"Found saved progress: stage {state['stage']}, {state['epoch']} epochs done")

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!kaggle datasets files abdallahalidev/plantvillage-dataset --page-size 1 > /dev/null && echo "Kaggle sign-in OK"

## 2. Download PlantVillage (colour + segmented) and PlantDoc

In [ ]:
import glob, pathlib

%cd /content
if not glob.glob('/content/pv/**/color', recursive=True):
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content
    !unzip -q -o /content/plantvillage-dataset.zip '*color/*' '*segmented/*' -d /content/pv
    os.remove('/content/plantvillage-dataset.zip')
if not os.path.isdir('/content/plantdoc'):
    !wget -q -O /content/plantdoc.zip https://github.com/pratikkayal/PlantDoc-Dataset/archive/refs/heads/master.zip
    !unzip -q -o /content/plantdoc.zip -d /content && mv /content/PlantDoc-Dataset-master /content/plantdoc && rm /content/plantdoc.zip

DATA_DIR = pathlib.Path(sorted(glob.glob('/content/pv/**/color', recursive=True))[0])
SEG_DIR  = DATA_DIR.parent / 'segmented'
PD_DIR   = pathlib.Path('/content/plantdoc')
print('PlantVillage colour:', DATA_DIR, '| segmented exists:', SEG_DIR.exists())
print('PlantDoc train folders:', len(list((PD_DIR / 'train').iterdir())), '| test folders:', len(list((PD_DIR / 'test').iterdir())))

## 3. Set aside the same 190 lab test photos as v2
Identical code and seed to v2, so both models are tested on exactly the same photos.

In [ ]:
import json, random
from collections import Counter

classes = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
TEST_DIR = pathlib.Path('/content/test_samples_v2')
PER_CLASS = 5
IMG_EXTS = {'.jpg', '.jpeg', '.png'}
manifest_path = TEST_DIR / 'manifest.json'

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    for item in manifest:
        (DATA_DIR / item['file']).unlink(missing_ok=True)
    print(f'Using the {len(manifest)} test photos already set aside.')
else:
    rng = random.Random(42)
    manifest = []
    for c in classes:
        imgs = sorted(p for p in (DATA_DIR / c).iterdir() if p.suffix.lower() in IMG_EXTS)
        for p in rng.sample(imgs, PER_CLASS):
            dest = TEST_DIR / c / p.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(p), dest)
            manifest.append({'file': f'{c}/{p.name}', 'label': c})
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'Moved {len(manifest)} test photos to {TEST_DIR}')
print(len(classes), 'classes')

## 4. Map PlantDoc classes onto the 38 PlantVillage classes
PlantDoc's plain "X leaf" folders are healthy leaves.

In [ ]:
PLANTDOC_TO_PV = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato leaf': 'Tomato___healthy',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf black rot': 'Grape___Black_rot',
    'grape leaf': 'Grape___healthy',
}
assert all(v in classes for v in PLANTDOC_TO_PV.values()), [v for v in PLANTDOC_TO_PV.values() if v not in classes]
CLASS_INDEX = {c: i for i, c in enumerate(classes)}

from PIL import Image

def list_plantdoc(split):
    items, skipped, bad = [], Counter(), 0
    for d in sorted((PD_DIR / split).iterdir()):
        if not d.is_dir(): continue
        if d.name not in PLANTDOC_TO_PV:
            skipped[d.name] += 1; continue
        for p in sorted(d.iterdir()):
            try:
                with Image.open(p) as im:
                    fmt = im.format
                    im.verify()                            # drop broken files
                if fmt not in ('JPEG', 'PNG', 'BMP', 'GIF'):   # formats TensorFlow can decode
                    raise ValueError(fmt)
                items.append((str(p), CLASS_INDEX[PLANTDOC_TO_PV[d.name]]))
            except Exception:
                bad += 1
    print(f'PlantDoc {split}: {len(items)} usable photos, {bad} broken files dropped, unmapped folders: {dict(skipped) or "none"}')
    return items

pd_train_all = list_plantdoc('train')
pd_test      = list_plantdoc('test')

## 5. The 12 Wikimedia Commons test photos (same as the v2 test)
Downloaded from Wikimedia Commons (free licenses), used **only for testing**.

In [ ]:
import time, urllib.request

WIKI_DIR = pathlib.Path('/content/wikimedia_test'); WIKI_DIR.mkdir(exist_ok=True)
T = 'https://thumb.wikimedia.org/wikipedia/commons/thumb/'
WIKI = [
    ('grape_black_rot_1.jpg', 'Grape___Black_rot', T + '1/14/Guignardia_bidwellii_%28black_rot%29_2.jpg/960px-Guignardia_bidwellii_%28black_rot%29_2.jpg'),
    ('grape_black_rot_2.jpg', 'Grape___Black_rot', T + '3/3f/Guignardia_bidwellii_%28black_rot%29_on_grape_1.jpg/960px-Guignardia_bidwellii_%28black_rot%29_on_grape_1.jpg'),
    ('tomato_late_blight_1.jpg', 'Tomato___Late_blight', T + 'e/eb/Late_blight_on_tomato_leaf_%287871756748%29.jpg/960px-Late_blight_on_tomato_leaf_%287871756748%29.jpg'),
    ('tomato_late_blight_2.jpg', 'Tomato___Late_blight', T + '4/43/Tomato_late_blight_leaf_sporulating_lesions_1_%285816170621%29.jpg/960px-Tomato_late_blight_leaf_sporulating_lesions_1_%285816170621%29.jpg'),
    ('potato_late_blight_1.jpg', 'Potato___Late_blight', T + 'a/aa/Late_blight_on_potato_leaf_2.jpg/960px-Late_blight_on_potato_leaf_2.jpg'),
    ('potato_late_blight_2.jpg', 'Potato___Late_blight', T + '9/9f/Potato_Late_Blight.JPG/960px-Potato_Late_Blight.JPG'),
    ('squash_powdery_mildew_1.jpg', 'Squash___Powdery_mildew', T + '5/59/K%C3%BCrbis-Cucurbita-Echter_Mehltau-Golovinomyces-4-JS.jpg/960px-K%C3%BCrbis-Cucurbita-Echter_Mehltau-Golovinomyces-4-JS.jpg'),
    ('squash_powdery_mildew_2.jpg', 'Squash___Powdery_mildew', T + '5/51/Powdery_mildew_9.jpg/960px-Powdery_mildew_9.jpg'),
    ('corn_common_rust_1.jpg', 'Corn_(maize)___Common_rust_', T + '2/27/Puccinia_sorghi_%2807%29.jpg/960px-Puccinia_sorghi_%2807%29.jpg'),
    ('corn_common_rust_2.jpg', 'Corn_(maize)___Common_rust_', T + 'f/fc/Puccinia_sorghi_%2811%29.jpg/960px-Puccinia_sorghi_%2811%29.jpg'),
    ('corn_northern_leaf_blight_1.jpg', 'Corn_(maize)___Northern_Leaf_Blight', T + '2/2d/Northern_corn_leaf_blight.JPG/960px-Northern_corn_leaf_blight.JPG'),
    ('strawberry_healthy_1.jpg', 'Strawberry___healthy', 'https://upload.wikimedia.org/wikipedia/commons/2/27/Two_strawberry_leaves_sunny.jpg'),
]
UA = {'User-Agent': 'PlantDiseaseCapstoneTest/1.0 (student project; github.com/ern352)'}
wiki_test = []
for name, label, url in WIKI:
    dest = WIKI_DIR / name
    for attempt in range(4):
        if dest.exists() and dest.stat().st_size > 1000: break
        try:
            dest.write_bytes(urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=30).read())
        except Exception as e:
            print('retrying', name, e); time.sleep(5 * (attempt + 1))
        time.sleep(2)
    if dest.exists(): wiki_test.append((str(dest), CLASS_INDEX[label]))
print(len(wiki_test), 'of 12 Wikimedia test photos ready')

## 6. Remove training photos that duplicate any test photo
Compares a 16×16 "difference hash" of every PlantDoc training photo with every test photo.

In [ ]:
import numpy as np

def dhash(path, size=16):
    with Image.open(path) as im:
        g = np.asarray(im.convert('L').resize((size + 1, size)), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

test_hashes = [dhash(p) for p, _ in pd_test + wiki_test]
test_hashes = np.stack(test_hashes)
pd_train_clean, removed = [], []
for p, y in pd_train_all:
    h = dhash(p)
    if (np.count_nonzero(test_hashes != h, axis=1) <= 25).any():   # <= 25 of 256 bits differ = near-duplicate
        removed.append(p)
    else:
        pd_train_clean.append((p, y))
print(f'Removed {len(removed)} PlantDoc training photos that duplicate a test photo; {len(pd_train_clean)} remain')

rng = random.Random(7)
rng.shuffle(pd_train_clean)
n_val = max(1, len(pd_train_clean) // 10)
pd_val, pd_train = pd_train_clean[:n_val], pd_train_clean[n_val:]
print(f'PlantDoc: {len(pd_train)} train · {len(pd_val)} validation (real-world, used for early stopping) · {len(pd_test)} test')

# Shrink the PlantDoc *training/validation* copies to 512 px once, so training doesn't decode huge photos
# thousands of times. Test photos stay untouched (full size, like the website receives them).
SMALL = pathlib.Path('/content/pd_small'); SMALL.mkdir(exist_ok=True)
def shrink(items):
    out = []
    for k, (p, y) in enumerate(items):
        dest = SMALL / f'{k}_{y}_{pathlib.Path(p).stem[:40]}.jpg'
        if not dest.exists():
            with Image.open(p) as im:
                im = im.convert('RGB'); im.thumbnail((512, 512)); im.save(dest, quality=92)
        out.append((str(dest), y))
    return out
pd_train = shrink(pd_train)
pd_val = shrink(pd_val)

## 7. Build the training data
PlantVillage 80/20 split (same seed as v2) + PlantDoc ×6. Each PlantVillage leaf is matched to its cut-out version for background swaps.

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'none')

IMG_SIZE = (224, 224)
BATCH_SIZE = 48
PD_REPEAT = 6

pv_items = [(str(p), CLASS_INDEX[c]) for c in classes for p in sorted((DATA_DIR / c).iterdir()) if p.suffix.lower() in IMG_EXTS]
rng = random.Random(123); rng.shuffle(pv_items)
n_pv_val = len(pv_items) // 5
pv_val, pv_train = pv_items[:n_pv_val], pv_items[n_pv_val:]

# segmented (cut-out) version of each PlantVillage photo, if it exists
seg_index = {}
if SEG_DIR.exists():
    for p in SEG_DIR.rglob('*'):
        if p.is_file():
            seg_index[(p.parent.name, p.stem.replace('_final_masked', ''))] = str(p)
def seg_for(path):
    p = pathlib.Path(path)
    return seg_index.get((p.parent.name, p.stem), '')
pv_train_seg = [seg_for(p) for p, _ in pv_train]
print(f'PlantVillage: {len(pv_train)} train · {len(pv_val)} validation · cut-outs matched for '
      f'{sum(bool(s) for s in pv_train_seg) / len(pv_train):.0%} of training photos')

bg_paths = [p for p, _ in pd_train]          # real backgrounds come only from PlantDoc *training* photos

train_paths = [p for p, _ in pv_train] + [p for p, _ in pd_train] * PD_REPEAT
train_segs  = pv_train_seg + [''] * (len(pd_train) * PD_REPEAT)
train_labels = [y for _, y in pv_train] + [y for _, y in pd_train] * PD_REPEAT
print(f'Training examples per epoch: {len(train_paths):,} ({len(pd_train) * PD_REPEAT:,} of them real-world)')

def decode(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(tf.cast(img, tf.float32), IMG_SIZE)

BG = tf.constant(bg_paths)
def random_background():
    bg = tf.io.decode_image(tf.io.read_file(BG[tf.random.uniform([], 0, len(bg_paths), tf.int32)]), channels=3, expand_animations=False)
    bg = tf.image.resize(tf.cast(bg, tf.float32), (int(IMG_SIZE[0] * 1.5), int(IMG_SIZE[1] * 1.5)))
    return tf.image.random_crop(bg, IMG_SIZE + (3,))

_g = np.exp(-np.linspace(-2, 2, 5) ** 2)
_k = np.outer(_g, _g); _k /= _k.sum()
GAUSS = tf.constant(_k[:, :, None, None].repeat(3, axis=2), tf.float32)   # 5x5 blur kernel, one per colour channel

def make_example(path, seg, label):
    img = decode(path)
    if tf.logical_and(tf.strings.length(seg) > 0, tf.random.uniform([]) < 0.5):
        cut = decode(seg)
        mask = tf.cast(tf.reduce_sum(cut, axis=-1, keepdims=True) > 40.0, tf.float32)
        img = cut * mask + random_background() * (1.0 - mask)
    if tf.random.uniform([]) < 0.3:               # slight blur, like a phone photo
        img = tf.nn.depthwise_conv2d(img[None], GAUSS, [1, 1, 1, 1], 'SAME')[0]
    return img, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (tf.data.Dataset.from_tensor_slices((train_paths, train_segs, train_labels))
            .shuffle(len(train_paths), seed=1, reshuffle_each_iteration=True)
            .map(make_example, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

def plain_ds(items):
    paths, labels = [p for p, _ in items], [y for _, y in items]
    return (tf.data.Dataset.from_tensor_slices((paths, labels))
            .map(lambda p, y: (decode(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))

pd_val_ds = plain_ds(pd_val)
pv_val_ds = plain_ds(pv_val)

In [ ]:
# Check the training images look right (background swaps, blur)
import matplotlib.pyplot as plt
xb, yb = next(iter(train_ds))
plt.figure(figsize=(18, 6))
for i in range(min(24, len(xb))):
    ax = plt.subplot(3, 8, i + 1); ax.imshow(xb[i].numpy().astype('uint8')); ax.axis('off')
    ax.set_title(classes[int(yb[i])].replace('___', ' ')[:22], fontsize=7)
plt.suptitle('Training examples (some PlantVillage leaves now have real backgrounds)'); plt.tight_layout(); plt.show()

## 8. Model: MobileNetV2 at 224 px with stronger augmentation

In [ ]:
from tensorflow.keras import layers

base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.RandomFlip('horizontal_and_vertical')(inputs)
x = layers.RandomRotation(0.2)(x)
x = layers.RandomZoom((-0.3, 0.1))(x)
x = layers.RandomTranslation(0.1, 0.1)(x)
x = layers.RandomBrightness(0.25)(x)
x = layers.RandomContrast(0.3)(x)
x = layers.Rescaling(1./127.5, offset=-1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
BASE_NAME = base.name
json.dump(classes, open(CKPT / 'class_names_v3.json', 'w'), indent=2)
print(f'{model.count_params():,} parameters')

### Checkpoint helpers
After every epoch: the full model (with optimizer) goes to `latest.keras`, the best epoch of each stage to
`best_stage1/2.weights.h5`, and the progress to `state.json`, all in Google Drive. Early stopping is done here too,
so it also survives a disconnect.

In [ ]:
class DriveCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, stage, patience):
        super().__init__(); self.stage, self.patience = stage, patience
    def on_epoch_end(self, epoch, logs=None):
        for k, v in (logs or {}).items():
            state['history'].setdefault(k, []).append(float(v))
        acc = float(logs['val_accuracy'])
        if acc > state['best']:
            state['best'], state['wait'] = acc, 0
            self.model.save_weights(str(CKPT / f'best_stage{self.stage}.weights.h5'))
        else:
            state['wait'] += 1
        state['epoch'] = epoch + 1
        self.model.save(str(CKPT / 'latest.keras'))
        save_state()
        print(f"  saved to Drive: stage {self.stage}, epoch {epoch + 1}, best real-world val {state['best']:.1%}")
        if state['wait'] >= self.patience:
            print('  early stopping'); self.model.stop_training = True

def train_stage(stage, epochs, patience):
    global model, base
    if state['stage'] > stage:
        print(f'Stage {stage} already finished earlier (saved in Drive).'); return
    if state['epoch'] > 0:
        model = tf.keras.models.load_model(str(CKPT / 'latest.keras')); base = model.get_layer(BASE_NAME)
        print(f"Resuming stage {stage} after epoch {state['epoch']}")
    if state['wait'] < patience and state['epoch'] < epochs:
        model.fit(train_ds, validation_data=pd_val_ds, initial_epoch=state['epoch'], epochs=epochs,
                  callbacks=[DriveCheckpoint(stage, patience)])
    model.load_weights(str(CKPT / f'best_stage{stage}.weights.h5'))  # keep the best epoch of this stage
    model.save(str(CKPT / f'after_stage{stage}.keras'))
    if stage == 1:
        state['stage1_epochs'] = len(state['history'].get('accuracy', []))
    state.update(stage=stage + 1, epoch=0, best=-1.0, wait=0); save_state()
    print(f'Stage {stage} done; best real-world validation accuracy restored.')

## 9. Train the head (stage 1)
Early stopping watches the **real-world validation photos** (PlantDoc), since that's what we want to improve.

In [ ]:
train_stage(1, epochs=6, patience=2)

## 10. Fine-tune (stage 2): unfreeze the last 60 layers, low learning rate

In [ ]:
if state['stage'] == 2 and state['epoch'] == 0:  # starting stage 2 now (not resuming in the middle of it)
    model = tf.keras.models.load_model(str(CKPT / 'after_stage1.keras')); base = model.get_layer(BASE_NAME)
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    # keep BatchNorm frozen (inference mode): a reloaded model forgets base(x, training=False)
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(2e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
train_stage(2, epochs=10, patience=3)
model = tf.keras.models.load_model(str(CKPT / 'after_stage2.keras'))

In [ ]:
h = state['history']
epochs = list(range(1, len(h['accuracy']) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(epochs, h['accuracy'], label='train (mixed)'); ax1.plot(epochs, h['val_accuracy'], label='real-world validation (PlantDoc)')
ax2.plot(epochs, h['loss'], label='train'); ax2.plot(epochs, h['val_loss'], label='real-world validation')
for ax, t in ((ax1, 'Accuracy'), (ax2, 'Loss')):
    ax.axvline(state['stage1_epochs'] + 0.5, ls='--', color='gray', label='fine-tuning starts')
    ax.set_title(t); ax.set_xticks(epochs); ax.set_xlabel('epoch'); ax.legend()
plt.show()
pv_val_loss, pv_val_acc = model.evaluate(pv_val_ds, verbose=0)
print(f'PlantVillage validation accuracy (lab photos, {len(pv_val):,}): {pv_val_acc:.1%}')

## 11. Compare v2 and v3 on the same three tests

In [ ]:
!wget -q -O /content/v2.keras https://huggingface.co/spaces/ERN-low/plant-disease-detector/resolve/main/plant_disease_model.keras
v2 = tf.keras.models.load_model('/content/v2.keras')
lab_test = [(str(TEST_DIR / m['file']), CLASS_INDEX[m['label']]) for m in manifest]
crop_of = lambda i: classes[i].split('___')[0]

def score(m, size, items, name):
    preds = []
    for i in range(0, len(items), 64):
        batch = tf.stack([tf.image.resize(tf.cast(tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False), tf.float32), size)
                          for p, _ in items[i:i + 64]])
        preds.append(m.predict(batch, verbose=0))
    probs = np.concatenate(preds); y = np.array([t for _, t in items])
    top3 = np.argsort(-probs, axis=1)[:, :3]
    return {'test': name, 'n': len(items),
            'top1': float((top3[:, 0] == y).mean()), 'top3': float((top3 == y[:, None]).any(1).mean()),
            'crop': float(np.mean([crop_of(a) == crop_of(b) for a, b in zip(top3[:, 0], y)])),
            'per_image': [{'file': p, 'true': classes[t], 'pred': classes[a], 'conf': round(float(probs[k, a]) * 100, 1),
                           'top3': [classes[j] for j in top3[k]]} for k, ((p, t), a) in enumerate(zip(items, top3[:, 0]))]}

tests = [('Lab: 190 PlantVillage photos', lab_test), (f'Real-world: PlantDoc test ({len(pd_test)})', pd_test), ('Real-world: 12 Wikimedia photos', wiki_test)]
report = {'v2': [score(v2, (160, 160), items, n) for n, items in tests],
          'v3': [score(model, IMG_SIZE, items, n) for n, items in tests],
          'plantvillage_val_acc_v3': float(pv_val_acc), 'removed_duplicates': len(removed),
          'pd_train': len(pd_train), 'pd_val': len(pd_val), 'pd_test': len(pd_test)}

print(f"{'Test':<38}{'v2 exact':>10}{'v3 exact':>10}{'v2 crop':>10}{'v3 crop':>10}{'v2 top3':>10}{'v3 top3':>10}")
for a, b in zip(report['v2'], report['v3']):
    print(f"{a['test']:<38}{a['top1']:>10.1%}{b['top1']:>10.1%}{a['crop']:>10.1%}{b['crop']:>10.1%}{a['top3']:>10.1%}{b['top3']:>10.1%}")
print('\nWikimedia photos, v3:')
for r in report['v3'][2]['per_image']:
    print(f"  {'✅' if r['true'] == r['pred'] else '❌'} {pathlib.Path(r['file']).name:<34} → {r['pred']} ({r['conf']}%)")
json.dump(report, open('/content/v3_comparison.json', 'w'), indent=2)

## 12. Save v3 (without optimizer state, so the file is small enough to upload easily)

In [ ]:
clean = tf.keras.models.clone_model(model)
clean.set_weights(model.get_weights())
sample = next(iter(pd_val_ds))[0][:8]
assert np.allclose(clean.predict(sample, verbose=0), model.predict(sample, verbose=0), atol=1e-4)
clean.save('/content/plant_disease_model_v3.keras')
json.dump(classes, open('/content/class_names_v3.json', 'w'), indent=2)
for f in ['plant_disease_model_v3.keras', 'class_names_v3.json', 'v3_comparison.json']:
    shutil.copy(f'/content/{f}', CKPT / f)
print(f"Saved plant_disease_model_v3.keras ({os.path.getsize('/content/plant_disease_model_v3.keras') / 1e6:.1f} MB)")
print('Copies of the model, class names and test report are in Google Drive:', CKPT)

## 13. Download the results
Also safe in Google Drive (`MyDrive/plant-disease-v3/`) if a download fails.

In [ ]:
for f in ['/content/plant_disease_model_v3.keras', '/content/class_names_v3.json', '/content/v3_comparison.json']:
    files.download(f)